# Image Classification with ResNet-18

Run a pretrained convolutional neural network (CNN) on a dog photo, then crop or rotate the photo and see how its five highest-scoring ImageNet labels change.

## 1. Set up the libraries

**PyTorch** runs the model, **TorchVision** provides the model and preprocessing, **Pillow** opens and edits images, and **Matplotlib** displays them. The next cell installs any missing libraries and reports the versions in use.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

import matplotlib
import matplotlib.pyplot as plt
import PIL
import torch
import torchvision
from PIL import Image
from torchvision.models import ResNet18_Weights, resnet18

print(f"PyTorch     {torch.__version__}")
print(f"TorchVision {torchvision.__version__}")
print(f"Pillow      {PIL.__version__}")
print(f"Matplotlib  {matplotlib.__version__}")

## 2. Download and open the sample image

The photo comes from the [official PyTorch Hub ResNet demo](https://pytorch.org/hub/pytorch_vision_resnet/). Pillow opens the JPEG and converts it to RGB so the model receives three color channels.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

image_url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
image_path = Path("resnet18_sample_dog.jpg")

with urlopen(image_url, timeout=30) as response:
    image_path.write_bytes(response.read())

with Image.open(image_path) as downloaded_image:
    image = downloaded_image.convert("RGB")

width, height = image.size
print(f"{width} × {height} pixels | {image.mode}")

fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.imshow(image)
ax.set_title("Sample input")
ax.axis("off")
plt.show()

## 3. Load pretrained weights and prepare the image

TorchVision downloads the ResNet-18 weights on first use and reuses its cached copy within the runtime. The weights were learned during training on ImageNet and are used as-is here.

[The preprocessing bundled with these weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html) resizes the shorter edge to 256 pixels, center-crops to 224 × 224, then converts and normalizes pixel values. The tensor starts with three dimensions (channels, height, width). `unsqueeze(0)` adds a batch dimension for the single image.

In [ ]:
device = torch.device("cpu")
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights).to(device)
model.eval()

preprocess = weights.transforms()

def prepare_batch(input_image):
    image_tensor = preprocess(input_image)
    return image_tensor.unsqueeze(0).to(device)

batch = prepare_batch(image)

print("Image tensor (channels, height, width):", tuple(batch.shape[1:]))
print("Model input (batch, channels, height, width):", tuple(batch.shape))
print("Device:", batch.device)

## 4. Run CPU inference and read the top five

ResNet-18 returns one raw score (logit) per ImageNet class. Softmax converts the 1,000 logits into scores that sum to 1. The table displays the five largest scores as percentages. The weights supply the class labels in the same order as the model outputs.

In [ ]:
labels = weights.meta["categories"]

def predict_top5(input_batch):
    # Inference mode skips gradient tracking.
    with torch.inference_mode():
        logits = model(input_batch)[0]
        probabilities = logits.softmax(dim=0)
        top_scores, top_indices = probabilities.topk(5)

    return [
        (labels[index], score)
        for index, score in zip(top_indices.tolist(), top_scores.tolist())
    ]

original_top5 = predict_top5(batch)

print(f"{'Rank':<6} {'ImageNet label':<24} {'Score':>8}")
print("-" * 40)
for rank, (label, score) in enumerate(original_top5, start=1):
    print(f"{rank:<6} {label:<24} {score:>8.2%}")

## 5. Experiment · Crop or rotate the photo

Change the controls in the next cell and rerun that cell. Each run starts from the original photo. The four crop controls remove a percentage from each edge. For example, setting `crop_bottom_pct` to 25 removes the bottom quarter. The remaining rectangle is then rotated. Positive angles turn counterclockwise, and negative angles turn clockwise.

Both images go through the same model preprocessing. The table below the images compares the five highest-scoring labels for each image at the same rank.

Try one change at a time:

1. **Baseline check.** Set all five controls to 0. Do the two columns match?
2. **Rotation only.** Keep the crop controls at 0 and try 90°, then 180°. Track the original top label’s rank and score.
3. **Crop only.** Set rotation to 0. Try to keep the dog's face in one crop, then leave it outside another. Compare the labels that appear.

In [ ]:
# @title Edit the image and compare predictions
crop_left_pct = 0  # @param {type:"slider", min:0, max:90, step:5}
crop_top_pct = 0  # @param {type:"slider", min:0, max:90, step:5}
crop_right_pct = 0  # @param {type:"slider", min:0, max:90, step:5}
crop_bottom_pct = 0  # @param {type:"slider", min:0, max:90, step:5}
rotation_degrees = 65  # @param {type:"slider", min:-180, max:180, step:5}

if crop_left_pct + crop_right_pct >= 100 or crop_top_pct + crop_bottom_pct >= 100:
    raise ValueError("Opposite crop percentages must add up to less than 100.")

width, height = image.size
# Pillow uses (left, top, right, bottom) pixel coordinates.
crop_box = (
    round(width * crop_left_pct / 100),
    round(height * crop_top_pct / 100),
    round(width * (100 - crop_right_pct) / 100),
    round(height * (100 - crop_bottom_pct) / 100),
)
cropped_image = image.crop(crop_box)
edited_image = cropped_image.rotate(
    rotation_degrees,
    resample=Image.Resampling.BICUBIC,
    expand=True,  # Enlarge the canvas to keep the rotated corners visible.
)
edited_top5 = predict_top5(prepare_batch(edited_image))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, title, photo in zip(axes, ("Original", "Edited"), (image, edited_image)):
    ax.imshow(photo)
    ax.set_title(f"{title} · {photo.width} × {photo.height} pixels")
    ax.axis("off")
plt.show()

print(f"{'Rank':<6} {'Original label':<25} {'Score':>7}    {'Edited label':<25} {'Score':>7}")
print("-" * 76)
for rank, (original, edited) in enumerate(zip(original_top5, edited_top5), start=1):
    original_label, original_score = original
    edited_label, edited_score = edited
    print(
        f"{rank:<6} {original_label:<25} {original_score:>7.2%}    "
        f"{edited_label:<25} {edited_score:>7.2%}"
    )